# Phase B: Time-Aware Preprocessing

This notebook validates time order, prepares an unfitted leakage-safe preprocessing transformer, and writes EDA figures. It does not fit or evaluate a predictive model.

In [ ]:
from pathlib import Path
import pandas as pd
import sys

project_root = Path.cwd().parent
if not (project_root / 'src').exists():
    project_root = Path.cwd()
sys.path.insert(0, str(project_root))

from src.data_preprocessing import (
    CATEGORICAL_FEATURES, FEATURE_COLUMNS, NUMERIC_FEATURES,
    build_preprocessor, chronological_split, create_eda_figures, load_and_validate_data,
)

data_path = project_root / 'EV_Battery_Charging_TR_Dataset_with_Notes (1).csv'
data = load_and_validate_data(data_path)
data.head()

In [ ]:
intervals = data['Timestamp'].diff().dropna()
print('Start:', data['Timestamp'].min())
print('End:', data['Timestamp'].max())
print('Interval counts:')
display(intervals.value_counts().sort_index())
print('Gaps > 1 minute:', (intervals > pd.Timedelta(minutes=1)).sum())

# Inspect rather than modify negative demand-current observations.
data.loc[data['DemandCurrent_A'] < 0, [
    'Timestamp', 'ChargingStage', 'DemandCurrent_A', 'ChargeCurrent_A', 'SOC_%', 'MaxTemp_C',
]]

In [ ]:
# This object is deliberately unfitted. Fit it only inside a later training pipeline.
linear_preprocessor = build_preprocessor(scale_numeric=True)
tree_preprocessor = build_preprocessor(scale_numeric=False)
X = data[FEATURE_COLUMNS]
y = data['MaxTemp_C']

train_data, test_data = chronological_split(data, train_fraction=0.8)
print('Selected numerical features:', NUMERIC_FEATURES)
print('Selected categorical features:', CATEGORICAL_FEATURES)
print('Training period:', train_data['Timestamp'].min(), 'to', train_data['Timestamp'].max(), len(train_data))
print('Test period:', test_data['Timestamp'].min(), 'to', test_data['Timestamp'].max(), len(test_data))

In [ ]:
figure_paths = create_eda_figures(data, project_root / 'results' / 'figures')
[str(path.relative_to(project_root)) for path in figure_paths]

## Guardrails retained for training

The target-adjacent thermal fields, risk labels, free text, raw timestamp, and identifiers remain excluded. The chronological test period must not be used to fit preprocessing or tune models.